# PE Geometry Probe R32

Seeds: `[42, 123]`

Damage efficiency is evaluated using the actual held-out `rho_rel_geometry`, not nominal target 0.03.

In [1]:

from pathlib import Path
import json
import os
import subprocess

PACKAGE = Path(r"/home/djoko.bandjur.ftnkm/Notebooks/FMLE_PE_GEOMETRY_PROBE_R32")
CONFIG = json.loads(
    (PACKAGE / "CONFIG_R32.json").read_text(encoding="utf-8")
)
SEEDS = [42, 123]

print("PACKAGE:", PACKAGE)
print("OUTPUT:", CONFIG["output_dir"])
print("SEEDS:", SEEDS)
print("RANDOM DIRECTIONS:", CONFIG["random_directions"])
print("DAMAGE EFFICIENCY:",
      CONFIG["damage_efficiency_denominator"])
print("CUDA_VISIBLE_DEVICES:",
      os.environ.get("CUDA_VISIBLE_DEVICES"))

subprocess.run(["nvidia-smi"], check=False)


PACKAGE: /home/djoko.bandjur.ftnkm/Notebooks/FMLE_PE_GEOMETRY_PROBE_R32
OUTPUT: /home/djoko.bandjur.ftnkm/Notebooks/results/pe_geometry_probe_n6_r32
SEEDS: [42, 123]
RANDOM DIRECTIONS: 32
DAMAGE EFFICIENCY: actual held-out rho_rel_geometry, not nominal target rho
CUDA_VISIBLE_DEVICES: None
Fri Jul 17 22:45:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.86.10              Driver Version: 570.86.10      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200               

CompletedProcess(args=['nvidia-smi'], returncode=0)

In [2]:

from pathlib import Path
import json
import os
import subprocess
import sys

PACKAGE = Path(r"/home/djoko.bandjur.ftnkm/Notebooks/FMLE_PE_GEOMETRY_PROBE_R32")
CONFIG = json.loads(
    (PACKAGE / "CONFIG_R32.json").read_text(encoding="utf-8")
)

CODE_DIR = Path(CONFIG["code_dir"])
OUTPUT_DIR = Path(CONFIG["output_dir"])
SEEDS = [42, 123]
TAG = "42_123"

env = os.environ.copy()
env["HOME"] = str(Path.home())
env["USER"] = env.get("USER") or "djoko.bandjur.ftnkm"
env["LOGNAME"] = env.get("LOGNAME") or env["USER"]
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONPATH"] = (
    str(CODE_DIR)
    + os.pathsep
    + env.get("PYTHONPATH", "")
)

cache_dir = PACKAGE / "cache" / TAG
cache_dir.mkdir(parents=True, exist_ok=True)

env["XDG_CACHE_HOME"] = str(cache_dir / "xdg")
env["TORCH_HOME"] = str(cache_dir / "torch")


def run_dataset(dataset):
    output_path = (
        OUTPUT_DIR / f"{dataset}_seeds_{TAG}.json"
    )

    common = [
        sys.executable,
        "-u",
        str(CODE_DIR / "mechanistic_geometry_probe.py"),
        "--dataset", dataset,
        "--scripts-dir", str(CODE_DIR),
        "--output-path", str(output_path),
        "--pe-types",
        "learned", "sinusoidal", "rope", "alibi",
        "--seeds", *[str(seed) for seed in SEEDS],
        "--target-rho", "0.03",
        "--random-directions", "32",
        "--gradient-images", "256",
        "--geometry-images", "64",
        "--rho-images", "32",
        "--num-workers", "4",
    ]

    if dataset == "imagenet":
        command = common + [
            "--models-dir", CONFIG["imagenet_models"],
            "--val-dir", CONFIG["imagenet_val"],
            "--gradient-batch", "8",
            "--geometry-batch", "4",
            "--rho-batch", "4",
        ]
    else:
        command = common + [
            "--models-dir", CONFIG["cifar_models"],
            "--val-dir", CONFIG["cifar_val"],
            "--gradient-batch", "64",
            "--geometry-batch", "16",
            "--rho-batch", "16",
        ]

    print("\n" + "=" * 78)
    print("COMMAND:")
    print(" ".join(command))
    print("=" * 78 + "\n")

    subprocess.run(
        command,
        cwd=str(PACKAGE),
        env=env,
        check=True,
    )

    return output_path


imagenet_output = run_dataset("imagenet")
cifar_output = run_dataset("cifar")

print("\nJOB COMPLETE")
print(imagenet_output)
print(cifar_output)



COMMAND:
/usr/bin/python -u /home/djoko.bandjur.ftnkm/Notebooks/FMLE_PE_GEOMETRY_PROBE_R32/code/mechanistic_geometry_probe.py --dataset imagenet --scripts-dir /home/djoko.bandjur.ftnkm/Notebooks/FMLE_PE_GEOMETRY_PROBE_R32/code --output-path /home/djoko.bandjur.ftnkm/Notebooks/results/pe_geometry_probe_n6_r32/imagenet_seeds_42_123.json --pe-types learned sinusoidal rope alibi --seeds 42 123 --target-rho 0.03 --random-directions 32 --gradient-images 256 --geometry-images 64 --rho-images 32 --num-workers 4 --models-dir /home/djoko.bandjur.ftnkm/Notebooks/ImageNet100_checkpoints --val-dir /home/djoko.bandjur.ftnkm/datasets/imagenet100/imagenet100_resized/val --gradient-batch 8 --geometry-batch 4 --rho-batch 4

LOCAL HELD-OUT PE GEOMETRY PROBE
dataset: imagenet
device: cuda
output: /home/djoko.bandjur.ftnkm/Notebooks/results/pe_geometry_probe_n6_r32/imagenet_seeds_42_123.json
target rho_rel: 0.03
gradient images: 256
geometry images: 64
rho calibration images: 32
random directions: 32


[1

In [3]:

from pathlib import Path
import json
import math

PACKAGE = Path(r"/home/djoko.bandjur.ftnkm/Notebooks/FMLE_PE_GEOMETRY_PROBE_R32")
CONFIG = json.loads(
    (PACKAGE / "CONFIG_R32.json").read_text(encoding="utf-8")
)

OUTPUT_DIR = Path(CONFIG["output_dir"])
SEEDS = [42, 123]
TAG = "42_123"

EXPECTED_DIRECTION_NAMES = (
    {"task_gradient"}
    | {f"random_{index:02d}" for index in range(32)}
)


def verify_dataset(dataset):
    path = OUTPUT_DIR / f"{dataset}_seeds_{TAG}.json"
    marker = path.with_suffix(".COMPLETE.json")

    if not path.is_file():
        raise FileNotFoundError(path)

    if not marker.is_file():
        raise FileNotFoundError(marker)

    payload = json.loads(path.read_text(encoding="utf-8"))

    for pe_type in ("learned", "sinusoidal", "rope", "alibi"):
        for seed in SEEDS:
            record = payload["results"][pe_type][str(seed)]

            assert record["status"] == "ok", (
                dataset, pe_type, seed, record.get("status")
            )

            directions = record["directions"]

            assert set(directions) == EXPECTED_DIRECTION_NAMES, (
                dataset,
                pe_type,
                seed,
                len(directions),
                sorted(set(directions) - EXPECTED_DIRECTION_NAMES),
                sorted(EXPECTED_DIRECTION_NAMES - set(directions)),
            )

            for direction_name, direction in directions.items():
                heldout = direction["heldout_geometry"]

                rho = float(heldout["rho_rel_geometry"])
                delta_ce = float(heldout["delta_ce"])
                stored_efficiency = float(
                    heldout[
                        "damage_efficiency_delta_ce_per_rho"
                    ]
                )

                assert math.isfinite(rho) and rho > 0, (
                    dataset, pe_type, seed, direction_name, rho
                )

                expected_efficiency = delta_ce / rho

                assert math.isclose(
                    stored_efficiency,
                    expected_efficiency,
                    rel_tol=1e-9,
                    abs_tol=1e-10,
                ), (
                    dataset,
                    pe_type,
                    seed,
                    direction_name,
                    stored_efficiency,
                    expected_efficiency,
                )

    print("VERIFIED:", path)


verify_dataset("imagenet")
verify_dataset("cifar")

print()
print("VERIFIED JOB SEEDS:", SEEDS)
print("Random directions per model: 32")
print("Total directions per model: 33")
print("Damage efficiency denominator: actual held-out rho")


VERIFIED: /home/djoko.bandjur.ftnkm/Notebooks/results/pe_geometry_probe_n6_r32/imagenet_seeds_42_123.json
VERIFIED: /home/djoko.bandjur.ftnkm/Notebooks/results/pe_geometry_probe_n6_r32/cifar_seeds_42_123.json

VERIFIED JOB SEEDS: [42, 123]
Random directions per model: 32
Total directions per model: 33
Damage efficiency denominator: actual held-out rho
